# Models analyses

In [74]:
"""Import models."""
import pickle
from pathlib import Path

xgboost_model_path = Path("../models/xgboost_model.pkl")
random_forest_model_path = Path("../models/random_forest_model.pkl")

xgb_model = pickle.loads(xgboost_model_path.read_bytes())
rf_model = pickle.loads(random_forest_model_path.read_bytes())

print(f"Loaded XGBoost model from {xgboost_model_path}")
print(f"Loaded Random Forest model from {random_forest_model_path}")


Loaded XGBoost model from ../models/xgboost_model.pkl
Loaded Random Forest model from ../models/random_forest_model.pkl


In [75]:
"""Import test sets."""
import numpy as np
import pandas as pd
from IPython.display import display

TESTING_PATH = "../data/testing/tennis_testing.xlsx"

test_set = pd.read_excel(TESTING_PATH)

target = "y"
y_test = test_set[target]

print(f"Loaded prepared test set with {len(test_set)} rows")


Loaded prepared test set with 16482 rows


### Baselines

In [76]:
def baseline_accuracy(name, prediction, valid_mask, y_test):
    """Accuracy of a prediction rule, computed only on matches where it applies."""
    accuracy = (prediction[valid_mask] == y_test[valid_mask]).mean()
    coverage = valid_mask.mean()
    print(f"{name}: {accuracy:.2%}  (coverage: {coverage:.1%})")
    return accuracy


# Majority class: whichever class (0 or 1) is more common in the test set.
maj_acc = y_test.value_counts(normalize=True).max()
print(f"Majority class: {maj_acc:.2%}")

valid_odds = test_set["Odds_1"].notna() & test_set["Odds_2"].notna() & (test_set["Odds_1"] != test_set["Odds_2"])
baseline_accuracy("Market (odds)", test_set["Odds_1"] < test_set["Odds_2"], valid_odds, y_test)

baseline_accuracy("Ranking", test_set["Rank_Log_Ratio"] > 0, test_set["Rank_Log_Ratio"].notna(), y_test)
baseline_accuracy("Points", test_set["Points_Log_Ratio"] > 0, test_set["Points_Log_Ratio"].notna(), y_test)


Majority class: 50.16%
Market (odds): 68.51%  (coverage: 97.1%)
Ranking: 63.65%  (coverage: 99.8%)
Points: 63.38%  (coverage: 99.8%)


np.float64(0.6338216521791988)

## Model Evaluation

In [77]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)


def get_model_data(model, test_set):
    model_features = list(model.feature_names_in_)
    X = test_set[model_features]
    y = test_set["y"]
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return model_features, X, y, proba, pred


def evaluate_model(name, model, test_set):
    model_features, X, y, proba, pred = get_model_data(model, test_set)

    print(f"{name}")
    print("Accuracy:", f"{accuracy_score(y, pred):.2%}")
    print("ROC AUC:", f"{roc_auc_score(y, proba):.4f}")
    print("Log loss:", f"{log_loss(y, proba):.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y, pred))

    return {
        "name": name,
        "features": model_features,
        "X": X,
        "y": y,
        "proba": proba,
        "pred": pred,
    }


xgb_results = evaluate_model("XGBoost", xgb_model, test_set)
rf_results = evaluate_model("Random Forest", rf_model, test_set)

agreement = (xgb_results["pred"] == rf_results["pred"]).mean()
prob_corr = pd.Series(xgb_results["proba"]).corr(pd.Series(rf_results["proba"]))

print(f"Model agreement: {agreement:.2%}")
print(f"Probability correlation: {prob_corr:.4f}")

XGBoost
Accuracy: 67.87%
ROC AUC: 0.7461
Log loss: 0.5991
Confusion matrix:
[[5699 2569]
 [2726 5488]]
Random Forest
Accuracy: 67.26%
ROC AUC: 0.7416
Log loss: 0.5965
Confusion matrix:
[[5639 2629]
 [2768 5446]]
Model agreement: 96.29%
Probability correlation: 0.9798


## Feature Usage

In [78]:
def source_feature_name(transformed_name, model_features):
    # Map transformed one-hot and missing-indicator names back to source columns.
    if transformed_name.startswith("num__missingindicator_"):
        return transformed_name.removeprefix("num__missingindicator_")
    if transformed_name.startswith("num__"):
        return transformed_name.removeprefix("num__")
    if transformed_name.startswith("cat__"):
        encoded = transformed_name.removeprefix("cat__")
    elif transformed_name.startswith("high_cardinality__"):
        encoded = transformed_name.removeprefix("high_cardinality__")
    else:
        return transformed_name

    for feature in model_features:
        if encoded.startswith(f"{feature}_"):
            return feature
    return encoded


def feature_usage(name, model, top_n=15):
    model_features = list(model.feature_names_in_)
    transformed_names = model[:-1].get_feature_names_out()
    estimator = model.named_steps["model"]

    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
        column = "Importance"
    else:
        values = np.abs(estimator.coef_[0])
        column = "Abs_Coefficient"

    usage = pd.DataFrame({"Feature": transformed_names, column: values})
    usage["Source_Feature"] = usage["Feature"].map(
        lambda feature: source_feature_name(feature, model_features)
    )

    usage_by_source = (
        usage.groupby("Source_Feature", as_index=False)[column]
        .sum()
        .sort_values(column, ascending=False)
    )

    print(f"{name} feature usage")
    display(usage_by_source.head(top_n))


feature_usage("XGBoost", xgb_model)
feature_usage("Random Forest", rf_model)


XGBoost feature usage


,Source_Feature,Importance
16,Surface_Elo_Diff,0.455088
9,Odds_Diff,0.326920
3,Elo_Diff,0.120956
11,Rank_Log_Ratio,0.097037
15,Surface,0.000000
14,Round,0.000000
13,Recent_Surface_Form_Diff,0.000000
12,Recent_Form_Diff,0.000000
10,Points_Log_Ratio,0.000000
0,Age_Diff,0.000000


Random Forest feature usage


,Source_Feature,Importance
9,Odds_Diff,0.340061
3,Elo_Diff,0.155027
16,Surface_Elo_Diff,0.154653
11,Rank_Log_Ratio,0.117571
10,Points_Log_Ratio,0.097087
2,Dominance_Form_Diff,0.042822
12,Recent_Form_Diff,0.018933
0,Age_Diff,0.016384
13,Recent_Surface_Form_Diff,0.013591
4,Fatigue_Diff,0.011887


## Most Confident Wrong Matches

In [79]:
def show_most_wrong(name, results, test_set, top_n=10):
    # Show confident errors: high confidence, wrong class.
    rows = pd.DataFrame({
        "Actual_Result": results["y"],
        "Predicted_Class": results["pred"].astype(int),
        "Prob_Player_1_Wins": results["proba"],
    }, index=test_set.index)

    rows["Actual_Winner"] = np.where(
        rows["Actual_Result"] == 1,
        test_set["Player_1"],
        test_set["Player_2"],
    )
    rows["Predicted_Winner"] = np.where(
        rows["Predicted_Class"] == 1,
        test_set["Player_1"],
        test_set["Player_2"],
    )
    rows["Confidence"] = np.where(
        rows["Predicted_Class"] == 1,
        rows["Prob_Player_1_Wins"],
        1 - rows["Prob_Player_1_Wins"],
    )

    wrong = rows[rows["Actual_Result"] != rows["Predicted_Class"]]
    wrong = wrong.sort_values("Confidence", ascending=False).head(top_n)

    print(f"{name}: most confident wrong matches")
    display(wrong.join(test_set))


pd.set_option("display.max_columns", None) # show all columns when displaying a dataframe

show_most_wrong("XGBoost", xgb_results, test_set)
show_most_wrong("Random Forest", rf_results, test_set)


XGBoost: most confident wrong matches


,Actual_Result,Predicted_Class,Prob_Player_1_Wins,Actual_Winner,Predicted_Winner,Confidence,Date,Player_1,Player_2,Odds_1,Odds_2,Rank_Log_Ratio,Points_Log_Ratio,Odds_Diff,Best_of_5,Age_Diff,Left_Handed_Matchup,Recent_Form_Diff,Recent_Surface_Form_Diff,Dominance_Form_Diff,Fatigue_Diff,Elo_Diff,Surface_Elo_Diff,H2H_Diff,H2H_Surface_Diff,Home_Diff,Court,Surface,Round,Tournament,y
15384,1,0,0.204342,Shevchenko A.,Shelton B.,0.795658,2026-03-21,Shevchenko A.,Shelton B.,6.5,1.11,-2.233592,-1.736589,-1.767442,0,1.859001,0,-0.4,-0.400000,-0.088214,0.716531,-205.614633,-213.702462,0,0,-1,Outdoor,Hard,2nd Round,Miami Open,1
12354,1,0,0.204342,Munar J.,Shelton B.,0.795658,2025-02-06,Munar J.,Shelton B.,7.0,1.10,-1.593934,-1.216689,-1.850600,0,5.429158,0,-0.2,-0.200000,0.014128,0.000000,-213.177118,-268.825668,-1,-1,-1,Indoor,Hard,2nd Round,Dallas Open,1
7028,1,0,0.204342,Brouwer G.,Rune H.,0.795658,2023-02-16,Brouwer G.,Rune H.,8.0,1.08,-2.877949,-2.142939,-2.002481,0,7.123888,0,0.0,0.066667,0.033328,-0.624297,-214.041855,-224.174616,0,0,1,Indoor,Hard,2nd Round,ABN AMRO World Tennis Tournament,1
10084,1,0,0.204342,Damm M.,Paul T.,0.795658,2024-03-23,Damm M.,Paul T.,7.0,1.10,-2.679063,-2.102326,-1.850600,0,NaN,0,0.2,0.200000,-0.066188,0.147083,-228.087456,-234.874206,0,0,0,Outdoor,Hard,2nd Round,Miami Open,1
12967,1,0,0.204342,Dedura Palomero D.,Shapovalov D.,0.795658,2025-04-15,Dedura Palomero D.,Shapovalov D.,8.0,1.08,-2.940803,-3.234633,-2.002481,0,-8.908966,0,0.1,0.100000,0.025577,-0.069483,-197.463193,-119.949653,0,0,1,Outdoor,Clay,1st Round,BMW Open,1
1883,1,0,0.204342,Ramos Vinolas A.,Schwartzman D.,0.795658,2021-02-26,Ramos Vinolas A.,Schwartzman D.,7.0,1.10,-1.652923,-1.055873,-1.850600,0,NaN,0,0.0,-0.200000,-0.021144,0.367879,-147.087208,-103.306206,-5,-5,-1,Outdoor,Clay,Quarterfinals,Cordoba Open,1
1930,1,0,0.204342,Lajovic D.,Medvedev D.,0.795658,2021-03-03,Lajovic D.,Medvedev D.,8.0,1.08,-2.197225,-1.646689,-2.002481,0,5.618070,0,-0.2,-0.200000,-0.068117,0.099661,-295.925857,-357.590248,0,-1,0,Indoor,Hard,1st Round,ABN AMRO World Tennis Tournament,1
12754,1,0,0.204342,Goffin D.,Alcaraz C.,0.795658,2025-03-22,Goffin D.,Alcaraz C.,15.0,1.03,-2.908721,-1.929122,-2.678491,0,12.407940,0,-0.2,-0.200000,-0.181683,0.151637,-336.880101,-278.437774,0,0,0,Outdoor,Hard,2nd Round,Miami Open,1
14405,1,0,0.204342,Griekspoor T.,Sinner J.,0.795658,2025-10-05,Griekspoor T.,Sinner J.,19.0,1.02,-2.740840,-1.944906,-2.924636,0,5.122519,0,-0.8,-0.800000,-0.211638,-0.846293,-333.894138,-348.549407,-4,-3,0,Outdoor,Hard,3rd Round,Shanghai Masters,1
12585,1,0,0.204342,Pacheco Mendez R.,Ruud C.,0.795658,2025-02-27,Pacheco Mendez R.,Ruud C.,13.0,1.04,-4.265493,-3.409554,-2.525729,0,-6.340862,0,-0.6,-0.600000,-0.235241,0.203114,-312.477123,-224.699751,0,0,1,Outdoor,Hard,2nd Round,Abierto Mexicano,1


Random Forest: most confident wrong matches


,Actual_Result,Predicted_Class,Prob_Player_1_Wins,Actual_Winner,Predicted_Winner,Confidence,Date,Player_1,Player_2,Odds_1,Odds_2,Rank_Log_Ratio,Points_Log_Ratio,Odds_Diff,Best_of_5,Age_Diff,Left_Handed_Matchup,Recent_Form_Diff,Recent_Surface_Form_Diff,Dominance_Form_Diff,Fatigue_Diff,Elo_Diff,Surface_Elo_Diff,H2H_Diff,H2H_Surface_Diff,Home_Diff,Court,Surface,Round,Tournament,y
15950,0,1,0.964823,Cerundolo J.M.,Sinner J.,0.964823,2026-05-28,Sinner J.,Cerundolo J.M.,1.01,26.00,4.025352,2.757451,3.248146,1,0.249144,0,0.6,0.6,0.194467,0.000000,543.267207,423.336648,1,0,0,Outdoor,Clay,2nd Round,French Open,0
10033,1,0,0.040939,Nardi L.,Djokovic N.,0.959061,2024-03-12,Nardi L.,Djokovic N.,17.00,1.03,-4.812184,-2.935181,-2.803655,0,-16.208077,0,-0.4,-0.6,-0.186521,0.000000,-618.660546,-595.256457,0,0,0,Outdoor,Hard,3rd Round,BNP Paribas Open,1
4589,1,0,0.044204,Vesely J.,Djokovic N.,0.955796,2022-02-24,Vesely J.,Djokovic N.,11.00,1.05,-4.812184,-2.779371,-2.349105,0,NaN,0,-0.2,-0.2,-0.100065,0.049787,-530.491373,-555.915824,0,-1,0,Outdoor,Hard,Quarterfinals,Dubai Tennis Championships,1
4584,0,1,0.948015,Gojowczyk P.,Zverev A.,0.948015,2022-02-24,Zverev A.,Gojowczyk P.,1.06,10.00,3.455265,2.365174,2.244316,0,-7.764545,0,0.6,0.6,0.241467,-0.035674,338.075004,310.551667,2,2,0,Outdoor,Hard,2nd Round,Abierto Mexicano,0
10545,1,0,0.053692,Tabilo A.,Djokovic N.,0.946308,2024-05-12,Tabilo A.,Djokovic N.,9.00,1.07,-3.465736,-2.023296,-2.129566,0,-10.031485,0,-0.2,-0.2,-0.134417,0.000000,-470.981685,-389.627019,0,0,0,Outdoor,Clay,3rd Round,Internazionali BNL d'Italia,1
7950,1,0,0.058170,Seyboth Wild T.,Medvedev D.,0.941830,2023-05-30,Seyboth Wild T.,Medvedev D.,11.00,1.05,-4.454347,-2.895280,-2.349105,1,-4.076660,0,-0.8,-0.8,-0.245395,-0.085461,-456.666800,-235.483806,0,0,0,Outdoor,Clay,1st Round,French Open,1
11542,0,1,0.941083,Van De Zandschulp B.,Alcaraz C.,0.941083,2024-08-30,Alcaraz C.,Van De Zandschulp B.,1.01,29.00,3.205453,2.297301,3.357345,1,-7.583847,0,0.4,0.2,0.113705,0.145538,389.583076,321.504187,2,2,0,Outdoor,Hard,2nd Round,US Open,0
6084,0,1,0.940761,Galan D.E.,Tsitsipas S.,0.940761,2022-08-30,Tsitsipas S.,Galan D.E.,1.04,13.00,2.933857,2.151272,2.525729,1,-2.149213,0,0.4,0.6,0.074432,0.099574,343.469119,328.877401,0,0,0,Outdoor,Hard,1st Round,US Open,0
12099,1,0,0.059342,Opelka R.,Djokovic N.,0.940658,2025-01-03,Opelka R.,Djokovic N.,11.00,1.05,-3.734262,-3.095399,-2.349105,0,-10.269678,0,-0.4,-0.4,-0.091775,0.000000,-404.370983,-390.770057,0,0,0,Outdoor,Hard,Quarterfinals,Brisbane International,1
15151,1,0,0.062525,Kypson P.,De Minaur A.,0.937475,2026-02-24,Kypson P.,De Minaur A.,10.00,1.06,-2.842970,-2.012781,-2.244316,0,-0.692676,0,-1.0,-1.0,-0.194949,-0.015978,-428.116497,-403.487677,0,0,0,Outdoor,Hard,1st Round,Abierto Mexicano,1
